## Description

This notebook includes the implementation of the second of the two main stages of the RAG pipeline: retrieval and generation. In addition to the implementation, we will test it using several standard types of questions

## Import libraries

In [ ]:
from qdrant_client.models import Filter, FieldCondition, MatchValue, Prefetch, SparseVector, FusionQuery
from fastembed import SparseTextEmbedding, TextEmbedding
from qdrant_client import QdrantClient
import cohere
from dotenv import load_dotenv
import os
from google import genai
import math
from IPython.display import Markdown, display

In [4]:
load_dotenv()

True

## RAG pipeline implementation

For retrieval from the vector database, we will use a hybrid search approach that combines vector and keyword search. To generate embeddings, we will use the same models that were used during data indexing: BAAI/bge-small-en-v1.5 and Qdrant/minicoil-v1. In addition to the standard RAG pipeline, we will employ techniques such as multi-query retrieval and reranking.

For reranking, we will use the Cohere Rerank API. For multi-query generation and final answer generation, we will use the Gemini API. To improve the understanding of user queries, we will also extract any relevant years mentioned in the query and apply year-based filtering before performing the hybrid search.

The implementation is divided into the following functions:

In [5]:
def get_dense_embedding(dense_model, text):

    return next(dense_model.embed([text])).tolist()

In [6]:
def get_sparse_embedding(sparse_model, text):

    sparse_embedding = next(sparse_model.embed([text]))

    values = sparse_embedding.values.tolist()
    indices = sparse_embedding.indices.tolist()

    return values, indices

In [7]:
def retrieval(query, qdrant_client, dense_model, sparse_model, relevant_years, limit_vector_search=200, limit_keywords_search=200, 
                                                                                                                        limit_result=100):

    dense_embedding = get_dense_embedding(dense_model, query)
    sparse_embedding_values, sparse_embedding_indices = get_sparse_embedding(sparse_model, query)


    query_filter = None

    if relevant_years != []:

        query_filter = Filter(
            should=[
                FieldCondition(
                    key="year",
                    match=MatchValue(value=year)
                )
                for year in relevant_years
            ]
        )


    results = qdrant_client.query_points(
        collection_name="regulations",
        with_payload=True,
        prefetch=[
            Prefetch(
                query=dense_embedding,
                using="dense",
                limit=limit_vector_search,
                filter=query_filter
            ),
            Prefetch(
                query=SparseVector(
                    indices=sparse_embedding_indices,
                    values=sparse_embedding_values
                ),
                using="sparse",
                limit=limit_keywords_search,
                filter=query_filter
            ),
        ],
        query=FusionQuery(fusion="rrf"),
        limit=limit_result,
    )


    return results.points

In [8]:
def reranking(query, list_points, cohere_client, top_n):

    texts = []

    for point in list_points:

        text = f"""
        Regulation Type: {point.payload['regulation_type']} regulation
        Year: {point.payload['year']}
        Article: {point.payload['article']}
        Chapter: {point.payload['chapter']}

        {point.payload['content']}
        """.strip()

        texts.append(text)


    response = cohere_client.rerank(
        model="rerank-v4.0-pro",
        query=query,
        documents=texts,
        top_n=top_n,
    )
    
    reranking_result = [ list_points[item.index] for item in response.results ]


    return reranking_result

In [9]:
def generate_multi_queries(query, gemini_client, number_generated_queries=4):

    gemini_prompt = f"""
    Generate {number_generated_queries} alternative search queries for the following question.

    The queries should have the same meaning but use different wording.

    Question:
    {query}

    Return only the queries, one per line.
    """

    response = gemini_client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=gemini_prompt
    )

    return [query] + response.text.split("\n")

In [10]:
def reciprocal_rank_fusion(result_lists, k=60):

    id_point = {}

    for points_list in result_lists:

        for idx, point in enumerate(points_list):

            id = point.id

            if id not in id_point:
                id_point[id] = point
                id_point[id].score = 0

            id_point[id].score += ( 1 / (k + (idx+1)) )


    rrf_result = sorted( list(id_point.items()), key=lambda x: x[1].score, reverse=True )
    rrf_result = list( dict(rrf_result).values() )


    return rrf_result     

In [27]:
def generate_answer(user_query, list_points, gemini_client):

    texts = []

    for point in list_points:

        text = f"""
        Regulation Type: {point.payload['regulation_type']} regulation
        Year: {point.payload['year']}
        Article: {point.payload['article']}
        Chapter: {point.payload['chapter']}

        {point.payload['content']}
        """.strip()

        texts.append(text)

    context = '\n\n'.join(texts)


    prompt = f"""
    You are an expert on FIA Formula 1 Regulations.

    Answer the question using ONLY the provided context.

    Rules:

    - Do not use outside knowledge.
    - If the answer is not available in the context, say:
    "The provided regulations do not contain enough information to answer this question."
    - For every factual statement, provide the corresponding year, regulation type, and article number.
    - If multiple articles are relevant, cite all of them.
    - Be precise, concise, and factual.
    - Prefer concise summaries over reproducing regulatory text.
    - Do not quote large portions of regulations unless necessary.
    - Use bullet points when comparing regulations.

    - When answering comparison questions, focus only on substantive regulatory changes.
    - Compare regulations based on the content and meaning of the rules, not article numbering.
    - Do not assume that two rules are equivalent because they share the same article number.
    - Do not assume that two rules are different because they have different article numbers.
    - Do not treat article renumbering, article references, cross-references, formatting changes, or wording changes as regulatory changes unless the actual meaning of the rule has changed.
    - Changes in article references, article numbering, and cross-references must not be reported as regulatory changes unless the provided context demonstrates a change in the underlying rule.
    - If the only observed difference is a change in article references, article numbering, or cross-references, classify the result as "Insufficient information for comparison" unless the content of the referenced regulations is available and demonstrates a substantive change.
    - A difference in wording or references alone is not sufficient evidence of a substantive regulatory change.
    - A regulatory change should be reported only when the rule's requirements, permissions, restrictions, procedures, obligations, or penalties have changed.
    - Before reporting a regulatory change, verify that the actual rule content differs between the years.

    - Structure comparison answers using the following sections:
    - Confirmed substantive regulatory changes
    - Confirmed unchanged regulations
    - Insufficient information for comparison

    - If no confirmed substantive regulatory changes are found, explicitly state:
    "No confirmed substantive regulatory changes were identified in the provided context."

    - Do not conclude that a rule was added, removed, or modified unless evidence from all relevant years is present in the context.
    - Missing articles in the retrieved context should be treated as insufficient information, not as evidence of a regulatory change.
    - If the available context is insufficient to compare a rule across years, classify it as "Insufficient information for comparison".

    - For comparison questions, summarize the outcome of the comparison before providing supporting details.
    - Focus on the most important findings rather than listing every matching article.

    Context:
    {context}

    Question:
    {user_query}
    """


    response = gemini_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )


    return response.text


In [28]:
def extract_years(query, gemini_client):

    prompt = f"""
    Extract Formula 1 regulation years mentioned in the question.

    If a range is requested, return all years in that range.

    Return ONLY a LIST of integers.

    Examples:

    Question:
    What were the Safety Car rules in 2021?

    Answer:
    [2021]

    Question:
    Compare the 2018 and 2026 regulations.

    Answer:
    [2018, 2026]

    Question:
    How did the regulations change from 2020 to 2026?

    Answer:
    [2020, 2021, 2022, 2023, 2024, 2025, 2026]

    Question:
    How did the regulations evolve between 2019 and 2022?

    Answer:
    [2019, 2020, 2021, 2022]

    Question:
    In which year was the fastest lap point removed?

    Answer:
    []

    Question:
    {query}
    """


    response = gemini_client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=prompt
    )


    return eval(response.text)

In [ ]:
def rag_pipeline(user_query, gemini_client, qdrant_client, dense_model, sparse_model, cohere_client):

    relevant_years = extract_years(user_query, gemini_client)

    queries = generate_multi_queries(user_query, gemini_client)

    hybrid_search_results_lists = [ retrieval(query, qdrant_client, dense_model, sparse_model, relevant_years) for query in queries]
    
    rrf_result = reciprocal_rank_fusion(hybrid_search_results_lists)


    reranking_top_n = 20
    reranking_result = []

    if len(relevant_years) <= 1:
        reranking_result = reranking(user_query, rrf_result, cohere_client, reranking_top_n)
    else:
        top_n_per_year = math.ceil( reranking_top_n / len(relevant_years) ) 

        for year in relevant_years:

            year_specific_points = list( filter( lambda x: x.payload['year'] == year, rrf_result ) )
            reranking_year_res = reranking(user_query, year_specific_points, cohere_client, min(top_n_per_year, len(year_specific_points)))

            reranking_result += reranking_year_res


    final_answer = generate_answer(user_query, reranking_result, gemini_client)


    return final_answer, reranking_result

## Retrieval and generation testing

After implementing the RAG pipeline, it is necessary to evaluate its performance. For testing purposes, five standard categories of questions commonly asked about Formula 1 regulations were selected. These question types cover factual retrieval, regulatory comparison, historical analysis, reverse lookup, and regulation traceability.

| Question Type | Example Question |
|---------------|------------------|
| Fact Retrieval | Under what conditions can the Safety Car be deployed during a race in 2024? |
| Comparison | How did the Safety Car regulations change between 2021 and 2022? |
| Historical Evolution | How have the Sprint regulations evolved from 2021 to 2026? |
| Reverse Lookup | In which year was the fastest lap point removed? |
| Regulation Traceability | Which articles in the 2026 Sporting Regulations govern Safety Car procedures? |

In [ ]:
# function to display the retrieved relevant chunks

def print_retrieved_chunks(retrieved_chunks):

    for i, point in enumerate(retrieved_chunks):       
        p = point.payload

        print("=" * 120)
        print(f"Result #{i+1}")
        print(f"Year       : {p['year']}")
        print(f"Article    : {p['article']}")
        print(f"Chapter    : {p['chapter']}")
        print("-" * 120)
        print(p['content'])
        print()
    

In [ ]:
# Initialize RAG pipeline components

dense_model = TextEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

sparse_model = SparseTextEmbedding(
    model_name="Qdrant/minicoil-v1"
)

qdrant_client = QdrantClient(path='../data/qdrant_storage')

cohere_client = cohere.ClientV2(api_key=os.getenv("COHERE_API_KEY"))

gemini_client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

### Question number 1: Fact Retrieval

In [45]:
user_query = "Under what conditions can the Safety Car be deployed during a race in 2024?"
answer, retrieved_chunks = rag_pipeline(user_query, gemini_client, qdrant_client, dense_model, sparse_model, cohere_client)

In [49]:
display(Markdown(answer))

The Safety Car may be deployed during a race in 2024 under the following conditions:

*   Upon the order of the clerk of the course (2024, sporting regulation, Article 55.3).
*   If Competitors or officials are in immediate physical danger on or near the track (2024, sporting regulation, Article 55.3).
*   The circumstances are not such as to necessitate suspending the sprint session or the race (2024, sporting regulation, Article 55.3).

In [47]:
print_retrieved_chunks(retrieved_chunks)

Result #1
Year       : 2024
Article    : 55.3
Chapter    : SAFETY CAR
------------------------------------------------------------------------------------------------------------------------
The safety car may be brought into operation to neutralise a sprint session or a race upon the order of the clerk of the course. It will be used only if Competitors or officials are in immediate physical danger on or near the track but the circumstances are not such as to necessitate suspending the sprint session or the race.

Result #2
Year       : 2024
Article    : 58.10
Chapter    : RESUMING A SPRINT SESSION OR A RACE
------------------------------------------------------------------------------------------------------------------------
The safety car will enter the pits after one (1) lap unless: a) The sprint session or the race is being resumed in wet conditions and the Race Director deems more than one lap is necessary, in which case the use of wet-weather tyres as specified under Article 30.

The system successfully identified the relevant regulation and returned a concise and accurate answer. The response correctly extracted the deployment conditions and provided precise references to the corresponding article and year. This demonstrates effective retrieval and citation capabilities for factual questions.

### Question number 2: Comparison

In [50]:
user_query = "How did the Safety Car regulations change between 2021 and 2022?"
answer, retrieved_chunks = rag_pipeline(user_query, gemini_client, qdrant_client, dense_model, sparse_model, cohere_client)

In [51]:
display(Markdown(answer))

The Safety Car regulations underwent several substantive changes between 2021 and 2022, primarily related to the procedure for the safety car's return after lapped cars overtake, and the terminology used for "sprint sessions". A new regulation detailing the purpose of safety car deployment was also introduced in 2022.

### Confirmed substantive regulatory changes

*   **Safety Car Return After Lapped Cars Overtake**
    *   In 2021, the safety car would return to the pits at the end of the following lap "once the last lapped car has passed the leader" (2021 Sporting Regulation Article 48.12).
    *   In 2022, the safety car returns to the pits at the end of the following lap "once the message “LAPPED CARS MAY NOW OVERTAKE” has been sent to all Competitors using the official messaging system" (2022 Sporting Regulation Article 55.13).
    *   This represents a change in the procedural trigger for the safety car's return after lapped cars have been permitted to overtake.

*   **End-of-Session Signal with Safety Car Deployed on Last Lap**
    *   In 2021, if the safety car was deployed on the last lap, cars would take the "end-of-race signal" as normal (2021 Sporting Regulation Article 48.15).
    *   In 2022, this was changed to "end-of- session signal", broadening the scope beyond just races (2022 Sporting Regulation Article 55.16).

*   **Terminology for Sprint Sessions in Safety Car Call-in Procedure**
    *   In 2021, the procedure for the withdrawal of yellow flags and display of green flags/lights at the Line, when the leader approaches, made an exception "on the last lap of the sprint qualifying session or the race" (2021 Sporting Regulation Article 48.13).
    *   In 2022, this exception was rephrased to "on the last lap of the sprint session or the race" (2022 Sporting Regulation Article 55.14).
    *   This change from "sprint qualifying session" to "sprint session" broadens the application of the rule.

*   **Terminology for Sprint Sessions in Laps Counted Under Safety Car**
    *   In 2021, each lap completed while the safety car was deployed was counted as a "sprint qualifying session lap or race lap" (2021 Sporting Regulation Article 48.14).
    *   In 2022, this was changed to a "sprint session lap or race lap" (2022 Sporting Regulation Article 55.15).
    *   Similar to the above, this change from "sprint qualifying session lap" to "sprint session lap" broadens the application.

*   **Explicit Statement on Safety Car Deployment Purpose**
    *   In 2022, a regulation was introduced explicitly stating that the safety car may be brought into operation to neutralise a sprint session or a race, and that it is to be used only if Competitors or officials are in immediate physical danger on or near the track without necessitating suspension of the session or race (2022 Sporting Regulation Article 55.3). No directly comparable regulation providing this explicit statement of purpose was found in the provided 2021 context.

### Confirmed unchanged regulations

*   **Safety Car Deployment Message and Signalling**
    *   The procedure for deploying the safety car, including the message "SAFETY CAR DEPLOYED" sent to Competitors via the official messaging system, the display of "SC" on FIA light panels, and waved yellow flags and "SC" boards at marshal's posts, remained unchanged (2021 Sporting Regulation Article 48.4, 2022 Sporting Regulation Article 55.4).

*   **Requirements for Cars Behind the Safety Car**
    *   The obligation for all competing cars to reduce speed, form up in line behind the safety car no more than ten car lengths apart, and maintain a minimum time in each marshalling sector and at safety car lines remained substantively unchanged (2021 Sporting Regulation Article 48.7, 2022 Sporting Regulation Article 55.7).

*   **General Prohibition on Overtaking Under Safety Car**
    *   The general rule prohibiting drivers from overtaking another car on the track, including the safety car, until passing the Line after the safety car returns to the pits, remained substantively unchanged. The listed exceptions for overtaking (e.g., if signalled by the safety car, when entering/leaving pits, in pit entry/lane/exit, car in garage, car with obvious problem) also remained substantively unchanged in their descriptions (2021 Sporting Regulation Article 48.8, 2022 Sporting Regulation Article 55.8).

*   **Leader's Position Behind Safety Car**
    *   The requirement for the leader to keep within ten car lengths of the safety car once behind it, except under specific circumstances, remained substantively unchanged (2021 Sporting Regulation Article 48.10, 2022 Sporting Regulation Article 55.10).

### Insufficient information for comparison

*   **Safety Car Position at the Start of Reconnaissance Lap**
    *   The cross-reference for an exception to the safety car covering a whole lap of the circuit changed from "Article 42.1c)" in 2021 to "Article 49)" in 2022 (2021 Sporting Regulation Article 48.2, 2022 Sporting Regulation Article 55.2). Without the content of these referenced articles, a substantive comparison cannot be made.

*   **Penalty Articles for Speed Infractions Under Safety Car**
    *   The referenced article for penalties changed from "Article 47.3a), b), c) or d)" in 2021 to "Article 54.3a), 54.3b), 54.3c) or 54.3d)" in 2022 (2021 Sporting Regulation Article 48.7, 2022 Sporting Regulation Article 55.7). The content of these penalty articles is not provided, making it impossible to confirm if the penalties themselves changed.

*   **Specific Overtaking Exceptions Under Safety Car**
    *   The cross-references for specific overtaking exceptions changed within the regulations (e.g., from "Articles 41.1c), 48.12, 51.6 and 51.12" to "Articles 49.6, 55.13, 58.6, and 58.12", and from "Article 48.11" to "Article 55.11") (2021 Sporting Regulation Article 48.8, 2022 Sporting Regulation Article 55.8). The content of these referenced articles is not provided, thus preventing a substantive comparison of these specific exceptions.

*   **Exceptions for Leader's Position Behind Safety Car**
    *   The cross-references for exceptions to the leader keeping within ten car lengths of the safety car changed from "Article 48.12 below" and "Article 48.13 below" in 2021 to "Article 55.12 below" and "Article 55.13 below" in 2022 (2021 Sporting Regulation Article 48.10, 2022 Sporting Regulation Article 55.10). The content of Articles 55.12 is not provided, making a full comparison impossible.

*   **Safety Car and Cars Using Pit Lane**
    *   The 2021 regulation detailing circumstances under which the clerk of the course may ask cars and the safety car to use the pit lane (2021 Sporting Regulation Article 48.11) does not have a direct content counterpart provided in the 2022 context. Although 2022 Article 55.8(g) references a 2022 Article 55.11, its content is not available for comparison.

*   **Exceptions for Laps Counted Under Safety Car**
    *   The cross-reference for the procedure to be followed in specific cases changed from "Article 41.1c)" in 2021 to "Article 49" in 2022 (2021 Sporting Regulation Article 48.14, 2022 Sporting Regulation Article 55.15). Without the content of these referenced articles, a substantive comparison cannot be made.

In [52]:
print_retrieved_chunks(retrieved_chunks)

Result #1
Year       : 2021
Article    : 48.12
Chapter    : SAFETY CAR
------------------------------------------------------------------------------------------------------------------------
If the clerk of the course considers it safe to do so, and the message "LAPPED CARS MAY NOW OVERTAKE" has been sent to all Competitors via the official messaging system, any cars that have been lapped by the leader will be required to pass the cars on the lead lap and the safety car. This will only apply to cars that were lapped at the time they crossed the Line at the end of the lap during which they crossed the first Safety Car line for the second time after the safety car was deployed. Having overtaken the cars on the lead lap and the safety car these cars should then proceed around the track at an appropriate speed, without overtaking, and make every effort to take up position at the back of the line of cars behind the safety car. Whilst they are overtaking, and in order to ensure this may be 

The system successfully identified both substantive regulatory changes and unchanged regulations. It correctly distinguished meaningful procedural modifications from simple article renumbering and grouped uncertain cases into an "Insufficient information" section. This demonstrates strong performance on regulatory comparison tasks.

### Question number 3: Historical Evolution

In [53]:
user_query = "How have the Sprint regulations evolved from 2021 to 2026?"
answer, retrieved_chunks = rag_pipeline(user_query, gemini_client, qdrant_client, dense_model, sparse_model, cohere_client)

In [54]:
display(Markdown(answer))

The Sprint regulations have evolved significantly from 2021 to 2026, primarily through the introduction of new regulations detailing the Sprint format, grid procedures, and permitted work during suspensions.

### Confirmed substantive regulatory changes

*   **Grid Formation for Sprint/Race:**
    *   **2021 (Sporting Regulation, Article 35.4):** The grid for the *race* was determined by the final classification of the sprint qualifying session.
    *   **2024 (Sporting Regulation, Article 42.2):** The grid for the *sprint session* was explicitly stated to be formed in accordance with sprint qualifying session results, with penalties for the sprint session applied.
    *   **2026 (Sporting Regulation, Article B2.3.4; Sporting Regulation, Article B2.2.1):** New comprehensive rules were introduced for forming the *Sprint* grid based on Sprint Qualifying. These include detailed procedures for applying penalties (e.g., cumulative unserved grid penalties over 12 months, with more than 15 penalties resulting in starting behind others), handling unclassified drivers, and defining an alternative grid formation based on the Drivers' Championship classification if Sprint Qualifying does not take place. Specific timelines for publishing provisional and final grids were also added.

*   **Personnel on Grid for Sprint Session:**
    *   **2022 (Sporting Regulation, Article 43.6):** A new regulation was introduced, establishing a limit of sixteen (16) team personnel per Competitor on the grid when the three-minute signal is shown for the sprint session.

*   **Work on Cars During Session Suspension:**
    *   **2022 (Sporting Regulation, Article 57.4b)viii compared to 2021 Sporting Regulation, Article 50.4b)viii):** The permission for "Repair of genuine accident damage" was modified to require that such damage be "as specified in Article 40.2u)", adding a specific definitional reference.
    *   **2024 (Sporting Regulation, Article 57.4a):** The specific articles referenced for calculating the length of sprint or race suspension added to the maximum time period were made more precise, distinguishing between Sprint Session suspension (Article 5.3a)iii)) and Race suspension (Article 5.4d)).
    *   **2025 (Sporting Regulation, Article 57.4b)iv; Sporting Regulation, Article 57.4b)v):** The restriction that changes to air ducts around front and rear brakes and changes to radiator ducts were allowed "during the race only" was removed, making these changes permissible during both suspended sprint sessions and races.
    *   **2025 (Sporting Regulation, Article 57.4b)x):** A new allowance was added to replenish or replace the cooling medium in the Driver Cooling System if a Heat Hazard is declared.

*   **Sprint Session-specific Maintenance:**
    *   **2023 (Sporting Regulation, Article 40.4):** New regulations were introduced allowing the replacement of specific components (e.g., brake friction material, engine exhaust system, engine oil filters, intake air filter, spark plugs) with same-specification parts during Competitions where a sprint session is scheduled. Replacement parts of different design were permitted if previously used in a qualifying session or a race, with prior FIA notification.

*   **Sprint Session Duration and Distance:**
    *   **2026 (Sporting Regulation, Article B2.3.3; Sporting Regulation, Article B2.3.2):** New comprehensive regulations were introduced defining the Sprint Session duration (1 hour, extended to 1.5 hours if suspended, with specific Safety Car start rules) and distance (least number of complete laps exceeding 100km, with Safety Car formation lap adjustments).

*   **Sprint Qualifying Session Format/Timing:**
    *   **2026 (Sporting Regulation, Article B2.2.1):** A new regulation was introduced specifying that Sprint Qualifying determines the Sprint starting grid and occurs on the first day of track running, with specific timing relative to FP1.

### Confirmed unchanged regulations

*   The prohibition of practice starts and the requirement for a tight formation during the formation lap (Sporting Regulation 2021, Article 37.9; Sporting Regulation 2022, Article 43.9; Sporting Regulation 2023, Article 43.9; Sporting Regulation 2024, Article 43.9; Sporting Regulation 2025, Article 43.9).
*   The principle that the sprint session, race, and timekeeping system do not stop when suspended, and the suspension length is added to the maximum time period (Sporting Regulation 2021, Article 50.4a; Sporting Regulation 2022, Article 57.4a; Sporting Regulation 2023, Article 57.4a; Sporting Regulation 2024, Article 57.4a; Sporting Regulation 2025, Article 57.4a).
*   The restriction that only team members, officials, and duly accredited television cameramen are permitted in the pit lane while the sprint session or race is suspended (Sporting Regulation 2021, Article 50.4c; Sporting Regulation 2022, Article 57.4c; Sporting Regulation 2023, Article 57.4c; Sporting Regulation 2024, Article 57.4c; Sporting Regulation 2025, Article 57.4c).
*   Specific types of work allowed on cars during suspension, such as starting the engine, adding compressed gases, fitting/removing cooling/heating devices, changes for driver comfort, changing wheels/tyres, and adjusting front wing aerodynamic setup without adding/removing/replacing parts (Sporting Regulation 2021, Article 50.4b)i), ii), iii), vi), vii), ix); Sporting Regulation 2022, Article 57.4b)i), ii), iii), vi), vii), ix); Sporting Regulation 2023, Article 57.4b)i), ii), iii), vi), vii), ix); Sporting Regulation 2024, Article 57.4b)i), ii), iii), vi), vii), ix); Sporting Regulation 2025, Article 57.4b)i), ii), iii), vi), vii), ix)).
*   The imposition of penalties for unnecessary overtaking during the resuming lap(s) (Sporting Regulation 2022, Article 58.10; Sporting Regulation 2025, Article 58.9).

### Insufficient information for comparison

*   **Pit Wall Personnel during Sprint Qualifying Start:** The 2021 regulation restricting pit wall personnel during the start of a sprint qualifying session (Sporting Regulation 2021, Article 37.12) is not present in the provided context for subsequent years, preventing a comparison to determine if it was removed, changed, or remained unchanged.
*   **Definition of Genuine Accident Damage (2022 to 2023):** While the reference article for "genuine accident damage" changed from Article 40.2u) in 2022 to Article 40.2 g) in 2023 (Sporting Regulation 2022, Article 57.4b)viii; Sporting Regulation 2023, Article 57.4b)viii), the content of these referenced articles is not provided. Therefore, it is impossible to determine if the actual meaning or scope of "genuine accident damage" changed.

In [55]:
print_retrieved_chunks(retrieved_chunks)

Result #1
Year       : 2021
Article    : 37.12
Chapter    : SPRINT QUALIFYING SESSION STARTING PROCEDURE
------------------------------------------------------------------------------------------------------------------------
Unless specifically authorised by the Safety Delegate, during the start of a sprint qualifying session the pit wall must be kept free of all persons with the exception of the team personnel permitted under Article 22.13, officials and fire marshals.

Result #2
Year       : 2021
Article    : 35.4
Chapter    : THE GRID
------------------------------------------------------------------------------------------------------------------------
a) The grid for the race will be drawn up based on the final classification of the sprint qualifying session with the driver finishing first on pole position. b) Drivers that are not classified in the sprint qualifying session will start at the rear of the grid in the order of the number of laps completed. Should more than one (1) d

The system effectively tracked regulatory evolution across multiple years. It identified major milestones, including the introduction of new Sprint procedures, grid formation rules, and suspension regulations. This confirms the system's ability to analyze long-term regulatory development.

### Question number 4: Reverse Lookup

In [56]:
user_query = "In which year was the fastest lap point removed?"
answer, retrieved_chunks = rag_pipeline(user_query, gemini_client, qdrant_client, dense_model, sparse_model, cohere_client)

In [57]:
display(Markdown(answer))

The provided regulations do not contain enough information to answer this question. In all provided years (2019, 2020, 2021, 2022, 2023, 2024), one point is explicitly stated to be awarded for the fastest lap of the race under specific conditions (2019 Sporting Regulation Article 6.4; 2020 Sporting Regulation Article 6.4; 2021 Sporting Regulation Article 6.4; 2022 Sporting Regulation Article 6.4; 2023 Sporting Regulation Article 6.4; 2024 Sporting Regulation Article 6.4).

In [58]:
print_retrieved_chunks(retrieved_chunks)

Result #1
Year       : 2019
Article    : 6.4
Chapter    : WORLD CHAMPIONSHIP
------------------------------------------------------------------------------------------------------------------------
Points for both titles will be awarded at each Event according to the following scale : 1st : 25 points 2nd : 18 points 3rd : 15 points 4th : 12 points 5th : 10 points 6th : 8 points 7th : 6 points 8th : 4 points 9th : 2 points 10th : 1 point In addition to the above, one point will be awarded to the driver who achieved the fastest lap of the race and to the constructor whose car he was driving, provided the lap time was achieved without incurring a penalty and he was in the top ten positions of the final race classification (see Article 45). No point will be awarded if the fastest lap is achieved by a driver who was classified outside the top ten positions.

Result #2
Year       : 2021
Article    : 6.4
Chapter    : WORLD CHAMPIONSHIP
---------------------------------------------------------

The system was unable to answer the question because the required information was not available in the retrieved regulations. However, it correctly identified the limitation of the available context and avoided generating unsupported information. This demonstrates reliable handling of incomplete knowledge but also highlights a limitation of the current dataset coverage.

### Question number 5: Regulation Traceability

In [59]:
user_query = "Which articles in the 2026 Sporting Regulations govern Safety Car procedures?"
answer, retrieved_chunks = rag_pipeline(user_query, gemini_client, qdrant_client, dense_model, sparse_model, cohere_client)

In [60]:
display(Markdown(answer))

The 2026 Sporting Regulations govern Safety Car procedures in the following articles:

*   **Deployment and General Operation:**
    *   Deployment of Safety Car (2026 Sporting Regulation B5.13.1)
    *   Rules during a Safety Car Deployment (2026 Sporting Regulation B5.13.2)
    *   Use of Pit Lane during a Safety Car Deployment (2026 Sporting Regulation B5.13.3)
    *   Order of Cars Behind the Safety Car (2026 Sporting Regulation B5.13.4)
    *   Duration of Safety Car Period (2026 Sporting Regulation B5.13.5)
    *   Withdrawal of Safety Car (2026 Sporting Regulation B5.13.6)
    *   Counting of laps completed while the Safety Car is deployed (2026 Sporting Regulation B5.13.7)
    *   Safety Car deployment during the last lap or at the beginning/end of the last lap (2026 Sporting Regulation B5.13.8)
    *   Conditions for Safety Car use in relation to Virtual Safety Car (2026 Sporting Regulation B5.12.5)

*   **Safety Car During Formation Laps:**
    *   Mandate for formation lap(s) behind the Safety Car and associated tyre requirements (2026 Sporting Regulation B5.10.1)
    *   Order and maximum allowable gap behind the Safety Car during formation laps (2026 Sporting Regulation B5.10.2)
    *   Procedure for F1 Cars starting from the Pit Lane during formation laps behind the Safety Car (2026 Sporting Regulation B5.10.4)
    *   Procedure for F1 Cars entering the Pit Lane during formation laps behind the Safety Car (2026 Sporting Regulation B5.10.5)
    *   Permitted overtaking during formation lap(s) behind the Safety Car (2026 Sporting Regulation B5.10.6)
    *   Shortening of TTCS due to formation laps behind the Safety Car (2026 Sporting Regulation B5.10.7)
    *   Procedures for a standing start after formation laps behind the Safety Car (2026 Sporting Regulation B5.10.8)
    *   Procedures for a rolling start after formation laps behind the Safety Car (2026 Sporting Regulation B5.10.9)

*   **Safety Car in Other Contexts:**
    *   Safety Car position at the front of the grid prior to reconnaissance lap(s) (2026 Sporting Regulation B5.2.1)
    *   Permitted overtaking during laps behind the Safety Car at the resumption of a session (2026 Sporting Regulation B5.15.3)

In [61]:
print_retrieved_chunks(retrieved_chunks)

Result #1
Year       : 2026
Article    : B5.13.4
Chapter    : TOTAL TIME CLASSIFIED SESSIONS (TTCS), Safety Car (SC)
------------------------------------------------------------------------------------------------------------------------
Order of Cars Behind the SC a. When instructed by the Race Director the green light on the Safety Car will be illuminated to signal to F1 Cars between it and the leader that they are required to pass. Once all such cars have passed the Safety Car the green light on the Safety Car will be extinguished to signal that overtaking is no longer permitted, with the exception of the cases listed in Article B5.13.2c. These F1 Cars will continue at reduced speed and without overtaking until they reach the queue of F1 Cars behind the Safety Car. b. If the Race Director considers track conditions are unsuitable for overtaking the message “OVERTAKING WILL NOT BE PERMITTED” will be sent to all Competitors. c. If the Race Director considers it safe to do so, the mess

The system successfully identified and aggregated multiple relevant articles related to Safety Car procedures. The response covered deployment, operation, pit lane usage, restarts, formation laps, and other Safety Car scenarios. This demonstrates effective retrieval and synthesis of information spread across multiple regulations.

## Overall testing conclusion

The testing results demonstrate that the proposed Formula 1 Regulatory Intelligence Assistant can successfully answer factual questions, compare regulations across different years, analyze long-term regulatory evolution, and identify relevant regulatory articles. The system provides transparent answers supported by article-level references and reliably reports cases where insufficient information is available. Overall, the developed RAG pipeline shows strong performance for regulatory search and analysis tasks. 